In [2]:
import torch
from torch import nn
from torch.nn import functional as F

In [3]:
net=nn.Sequential(nn.Linear(20,256),nn.ReLU(),nn.Linear(256,10))
# 定义了一个特殊的 module-->相当于一个层 or 神经网络

In [ ]:
class MySequential(nn.Module):
    def __init__(self,*args):
        super().__init__()
        for block in args:
            self._modules[block]=block
    
    def forward(self,X):
        for block in self._modules.values():
            X =block(X)
        return X

## 1. __init__(self, *args)
- *args：把传进来的所有层收成一个元组，所以可以塞任意多层：
  MySequential(lin1, relu, lin2, relu, lin3)
- super().__init__()：调用 nn.Module 的初始化，搭好底层框架。
- for block in args: self._modules[block] = block
  把每层登记进 _modules。关键点：只有登记进 _modules 的子模块，
  nn.Module 才会"认识"它们——参数、.to()、.train()/.eval()、保存权重才管用。
  （标准写法其实是 self.add_module(str(i), block)，用模块当 key 能跑但不够规范）

## 2. forward(self, x)
- 按 _modules 里的顺序，把 X 依次送进每个 block：
  X = block(X)  →  上一层的输出，就是下一层的输入。
- 最后 return X，即走完所有层后的结果。

## 3. 一句话本质
它就是 nn.Sequential 的"手搓版"：*args 收层 + _modules 登记 + forward 里排队执行。

运算：y = x·Wᵀ + b

权重 W 形状 (256, 20)

偏置 b 形状 (256,)

In [4]:
X = torch.randn(2,20)
print(net(X))

tensor([[ 0.1304,  0.0681,  0.1555, -0.2063, -0.0855, -0.3364,  0.0031, -0.0762,
          0.1212,  0.1010],
        [ 0.1511,  0.3151,  0.4906,  0.1008, -0.0325,  0.1558,  0.1130,  0.2471,
          0.1747, -0.1214]], grad_fn=<AddmmBackward0>)


当你写 nn.Linear(20, 256) 时，PyTorch 在内部 __init__ 里已经做了这些事：

创建权重 W，形状 (256, 20)

创建偏置 b，形状 (256,)

用一套默认方案给它们填上随机值（Linear 默认是 kaiming_uniform，bias 用均匀随机）


我们也可以自定义某一层

每一层都是nn.Module的一个子类

In [7]:
class NestMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(20,64),nn.ReLU(),nn.Linear(64,32),nn.ReLU())
        
        nn.linear=nn.Linear(32,16)
    def forward(self,X):
        return nn.linear(self.net(X))
chimera= nn.Sequential(NestMLP())
print(chimera(X))

tensor([[-0.1203,  0.0408,  0.0861,  0.2010, -0.0993,  0.1956, -0.1234, -0.1375,
         -0.1879, -0.1678, -0.0938, -0.0685, -0.1261,  0.1399,  0.0586,  0.1622],
        [-0.1472,  0.0040, -0.0448,  0.0443,  0.0104,  0.1137, -0.1123, -0.1057,
         -0.1406, -0.0886, -0.0975, -0.0410,  0.0003,  0.2104,  0.0884,  0.1750]],
       grad_fn=<AddmmBackward0>)
